# A1 — Corpus Cleaning

In [1]:
import os
import re
import json
import pandas as pd

BASE_DIR = "/Users/I771657/Library/CloudStorage/OneDrive-SAPSE/Documents/School/MINI"
LYRICS_DIR = os.path.join(BASE_DIR, "songs", "songs_lyrics")
EXCEL_PATH = os.path.join(BASE_DIR, "songs", "songs_summary.xlsx")
DATA_DIR = os.path.join(BASE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
df_meta = pd.read_excel(EXCEL_PATH)
df_meta.columns = ["song_id", "title", "year", "album"]
print(f"Loaded {len(df_meta)} songs from Excel")
print(df_meta.head())
print(df_meta["year"].value_counts().sort_index())

Loaded 167 songs from Excel
   song_id             title  year        album
0        1       אותיות נחמה  2025  אותיות נחמה
1        2  משמעות לחלומותיי  2025  אותיות נחמה
2        3       כשהכל יתבהר  2025  אותיות נחמה
3        4       היה לי אותך  2025  אותיות נחמה
4        5         פעם פם פם  2025  אותיות נחמה
year
1970    12
1975    22
1978     8
1979     8
1988    15
1992    12
1996    19
2002    15
2007    15
2012    12
2016    14
2025    15
Name: count, dtype: int64


In [3]:
def load_lyrics(song_id: int, suffix: str = "") -> str:
    """Load lyrics file. suffix e.g. '_translated' for the clean Hebrew version."""
    filename = f"{song_id:03d}{suffix}.txt"
    path = os.path.join(LYRICS_DIR, filename)
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return f.read()

# Songs that have a translated (all-Hebrew) version for DictaBERT
TRANSLATED_IDS = set()
for fname in os.listdir(LYRICS_DIR):
    if "translat" in fname.lower():
        # extract song_id from filenames like 023_translated.txt or 061.translated.txt
        sid = int(fname.split("_")[0].split(".")[0])
        TRANSLATED_IDS.add(sid)

print(f"Songs with translated versions: {sorted(TRANSLATED_IDS)}")

Songs with translated versions: [23, 61, 138]


In [4]:
def normalize_text(text: str) -> str:
    """Normalize a Hebrew lyrics string: punctuation, whitespace, blank lines."""
    # Normalize special punctuation
    text = text.replace('\u2013', '-')   # en-dash → hyphen
    text = text.replace('\u2014', '-')   # em-dash → hyphen
    text = text.replace('\u00b4', "'")  # acute accent → apostrophe
    text = text.replace('~', '')

    # Remove Unicode directional marks (U+200E, U+200F)
    text = re.sub(r'[\u200e\u200f]', '', text)

    # Collapse multiple spaces/tabs on a single line to one space
    text = re.sub(r'[ \t]+', ' ', text)

    # Strip trailing space from each line
    text = '\n'.join(line.strip() for line in text.split('\n'))

    # Collapse 3+ consecutive newlines to exactly 2 (stanza separator)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [5]:
corpus = []
issues = []

for _, row in df_meta.iterrows():
    song_id = int(row["song_id"])
    title = str(row["title"])

    text_raw = load_lyrics(song_id)
    if text_raw is None:
        issues.append({"song_id": song_id, "title": title, "issue": "file_not_found"})
        continue

    text_clean = normalize_text(text_raw)

    # For DictaBERT: use translated version if available, otherwise same as text_clean
    if song_id in TRANSLATED_IDS:
        # try both naming conventions: 023_translated.txt and 061.translated.txt
        text_raw_dicta = load_lyrics(song_id, '_translated') or load_lyrics(song_id, '.translated')
        if text_raw_dicta:
            text_dicta = normalize_text(text_raw_dicta)
            issues.append({"song_id": song_id, "title": title, "issue": "has_translated_version",
                           "action": "text_dicta uses translated file"})
        else:
            text_dicta = text_clean
    else:
        text_dicta = text_clean

    corpus.append({
        "song_id": song_id,
        "title": title,
        "year": int(row["year"]),
        "album": str(row["album"]),
        "text_clean": text_clean,
        "text_dicta": text_dicta,
        "word_count": len(text_clean.split()),
        "line_count": len([l for l in text_clean.split('\n') if l.strip()]),
    })

print(f"Corpus built: {len(corpus)} songs")
print(f"Issues logged: {len(issues)}")
for issue in issues:
    print(f"  Song {issue['song_id']:03d} ({issue['title']}): {issue['issue']}")

Corpus built: 167 songs
Issues logged: 3
  Song 023 (פעם אחת בחודש): has_translated_version
  Song 061 (להציל אותך): has_translated_version
  Song 138 (הריקוד האחרון של 1956): has_translated_version


In [6]:
corpus_path = os.path.join(DATA_DIR, "corpus.json")
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print(f"Saved corpus to {corpus_path}")
print(f"File size: {os.path.getsize(corpus_path) / 1024:.1f} KB")

Saved corpus to /Users/I771657/Library/CloudStorage/OneDrive-SAPSE/Documents/School/MINI/data/corpus.json
File size: 596.4 KB


In [7]:
issues_path = os.path.join(DATA_DIR, "issues_report.json")
with open(issues_path, "w", encoding="utf-8") as f:
    json.dump(issues, f, ensure_ascii=False, indent=2)

print(f"Saved issues report to {issues_path}")

Saved issues report to /Users/I771657/Library/CloudStorage/OneDrive-SAPSE/Documents/School/MINI/data/issues_report.json


In [8]:
df_corpus = pd.DataFrame(corpus)
print("=== Corpus Summary ===")
print(f"Total songs:        {len(df_corpus)}")
print(f"Albums:             {df_corpus['album'].nunique()}")
print(f"Year range:         {df_corpus['year'].min()} – {df_corpus['year'].max()}")
print(f"Avg words/song:     {df_corpus['word_count'].mean():.0f}")
print(f"Min words/song:     {df_corpus['word_count'].min()} (song {df_corpus.loc[df_corpus['word_count'].idxmin(), 'song_id']})")
print(f"Max words/song:     {df_corpus['word_count'].max()} (song {df_corpus.loc[df_corpus['word_count'].idxmax(), 'song_id']})")
print()
print("Songs per album:")
print(df_corpus.groupby(["year", "album"])["song_id"].count().to_string())

=== Corpus Summary ===
Total songs:        167
Albums:             13
Year range:         1970 – 2025
Avg words/song:     188
Min words/song:     66 (song 150)
Max words/song:     382 (song 37)

Songs per album:
year  album          
1970  שלמה ארצי          12
1975  את ואני            10
      משחקי 26           12
1978  גבר הולך לאיבוד     8
1979  דרכים               8
1988  חום יולי אוגוסט    15
1992  ירח                12
1996  שניים              19
2002  צימאון             15
2007  שפויים             15
2012  אושר אקספרס        12
2016  קצפת               14
2025  אותיות נחמה        15
